# Demo 4: Role-Based Briefing with Actor Context
This demo shows how different roles (engineering vs. leadership) see different memory summaries based on their read context.
## The Real-World Problem

**Scenario:** A production incident happens. Everyone gets the same raw incident data, but they need DIFFERENT insights:

| Role | What They Need | Current Problem |
|------|----------------|-----------------|
| **Support Lead** | Customer impact, ETA, communication template | Has to dig through technical details |
| **Engineer** | Root cause, debug logs, patches needed | Buried in business context |
| **Finance/Leadership** | Revenue impact, customer risk, decision timeline | No context on strategic implications |

**The Challenge:**
- ❌ Without Ninai: You manually write 3 separate briefings for each role from the same incident
- ✅ With Ninai: One API call + role input = tailored briefing for each person

**What Ninai Does (using Cognitive Gateway):**
1. **Support briefing:** "3 customers affected, ETA 30 min, send this message"
2. **Engineering briefing:** "OAuth mismatch in production, patch in this repo, test like this"
3. **Leadership briefing:** "~$50K/hour ARR exposure, recommend declare P1, notify board"

Same incident. Three intelligent, actionable briefings. Generated in seconds.

---

## Learning Goals

1. **Understand the problem** — Same data, different role needs = manual work
2. **Create incident memories** — Multi-faceted incident data
3. **Use Cognitive Gateway** — Delegate role-specific planning to backend agents
4. **Get tailored briefings** — Each role gets what they need, not a data dump
5. **Realize smart defaults** — Ninai knows what each role cares about without explicit instructions

## What You'll Learn

- ✓ Write multi-faceted incident data to Ninai
- ✓ Call `client.cognitive.gateway.plan()` with role parameters
- ✓ Receive role-specific action plans
- ✓ See how Ninai tailors analysis by context (role)
- ✓ Get structured output: priorities, actions, metrics for each role

**One incident → Three personalized, actionable briefings.** 🎯

In [ ]:
## Step 1: Setup and Login


from ninai import NinaiClient
import uuid

BASE_URL = 'https://admin.ninai.sansten.com/api/v1'
EMAIL = 'demo@ninai.dev'
PASSWORD = 'demo1234'
ORG_SLUG = 'default'

client = NinaiClient(base_url=BASE_URL)
client.login(email=EMAIL, password=PASSWORD, org_slug=ORG_SLUG)
seed = str(uuid.uuid4())[:8]

print(f"OK: Authenticated")
print(f"Seed: {seed}")
print(f"Ready for Step 2")

In [ ]:
## Step 2: Create Multi-Faceted Incident Memories

**Important:** Do NOT re-run Step 1 after this point. The seed will change.

Create incident data covering multiple dimensions: technical, business, customer impact.

print("=" * 80)
print("STEP 2: Creating Incident Data (Multi-Faceted)")
print("=" * 80)
print(f"\nSeed: {seed}")
print("DO NOT re-run Step 1 or the seed will change!\n")

# Create incident memories with multiple facets
incident_data = [
    f"TECHNICAL: Production outage - ACME auth service unreachable. OAuth2 token endpoint returning 500. Root cause: audience mismatch in JWT validation (prod config != staging). 9 hours duration. seed={seed}",
    
    f"CUSTOMER IMPACT: 127 customers affected. 9,000+ failed login attempts. ACME (priority customer) down 9 hours. Estimated ARR exposure: $45K. SLO breach: 99.95% down to 98.2%. seed={seed}",
    
    f"SUPPORT STATUS: 34 support tickets created. Top requests: refund, account credit, status updates. Communications sent: 4 (T+1h, T+3h, T+6h, T+9h). ETA fix: 30 minutes once patch deploys. seed={seed}",
    
    f"ENGINEERING NOTES: OAuth audience validation added 2 weeks ago. Config drift: staging has correct audience value, prod does not. Patch ready: change JWT_AUDIENCE env var. No database migration needed. Validation: run auth-flow-e2e suite. seed={seed}",
]

for idx, data in enumerate(incident_data, 1):
    mem = client.memories.create(
        content=data,
        source_type='manual',
        tags=['incident', 'p1', 'outage', seed]
    )
    print(f"Created memory {idx}/{len(incident_data)}: {data.split(':')[0]}")

print(f"\n✓ Total memories: {len(incident_data)}")
print(f"✓ Tagged with seed: {seed}")
print(f"\nProceed to Step 3")

## Step 3: Generate Role-Based Briefings

Run the next cell to invoke Cognitive Gateway for each role.

Each call uses `client.cognitive.gateway.plan()` with a different role context. Ninai analyzes the incident through that role's lens and returns prioritized, actionable briefings.

Expected outcome:
- 3 role-based briefings generated
- Each tailored to Support, Engineering, and Leadership needs
- Support sees: customer impact, communication strategy, ETA
- Engineering sees: root cause, patch location, testing steps
- Leadership sees: business impact, risk level, escalation decision

In [ ]:
# Step 3: Generate Role-Based Briefings via Cognitive Gateway
print("=" * 80)
print("STEP 3: Role-Based Briefing Generation")
print("=" * 80)
print(f"\nAnalyzing incident with seed: {seed}\n")

# Retrieve incident data
search_results = client.memories.search(query=f'seed={seed}', limit=20, hybrid=True)
print(f"Retrieved {len(search_results.items)} incident memories:")
for mem in search_results.items:
    facet = mem.content_preview.split(':')[0]
    print(f"  - {facet}...")

if not search_results.items:
    print("\nERROR: No memories found.")
    print("Check Step 2 or ensure you did NOT re-run Step 1")
else:
    # Combine all incident data
    combined_incident = "\n\n".join([mem.content_preview for mem in search_results.items])
    
    # Define roles and their focus areas
    roles = {
        'support': 'customer communication strategy, affected customers, ETA, talking points',
        'engineering': 'root cause details, technical patch, testing steps, deployment safety',
        'leadership': 'business impact, financial exposure, escalation decision, ARR risk'
    }
    
    briefings = {}
    
    print("\n" + "=" * 80)
    print("INVOKING COGNITIVE GATEWAY FOR EACH ROLE")
    print("=" * 80)
    
    for role, focus in roles.items():
        try:
            print(f"\n[{role.upper()}] Generating briefing...")
            print(f"  Focus areas: {focus}\n")
            
            # Call Cognitive Gateway with role context
            plan_result = client.cognitive.gateway.plan(
                goal=f"Create actionable incident briefing for {role}",
                context={
                    "incident_data": combined_incident,
                    "role": role,
                    "focus_areas": focus,
                    "output_format": "structured_action_plan"
                }
            )
            
            briefings[role] = plan_result
            
            # Display results
            print(f"✓ Briefing generated for {role}")
            print(f"\nPriority: {plan_result.get('priority', 'N/A')}")
            print(f"Confidence: {plan_result.get('confidence', 0):.2%}")
            
            # Show actions
            actions = plan_result.get('action_plan', [])
            if actions:
                print(f"\nKey Actions ({len(actions)}):")
                for idx, action in enumerate(actions[:5], 1):
                    if isinstance(action, dict):
                        print(f"  {idx}. {action.get('action', str(action))}")
                    else:
                        print(f"  {idx}. {action}")
            
            # Show metrics/focus
            focus_items = plan_result.get('focus_metrics', {})
            if focus_items:
                print(f"\nKey Metrics:")
                for key, value in list(focus_items.items())[:3]:
                    print(f"  • {key}: {value}")
        
        except AttributeError as e:
            print(f"ERROR: SDK method not available: {e}")
            print("Fix: Ensure SDK version 0.1.0+ is installed")
        except Exception as e:
            print(f"ERROR: {e}")
            print("Troubleshooting:")
            print("  - Backend running?")
            print("  - /cognitive/gateway/plan endpoint exists?")
            print("  - Auth token valid?")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("""
Same incident data → Three personalized briefings:

✓ SUPPORT BRIEFING:
  - Customer impact: 127 affected
  - Communication: 4 updates sent, next ETA-based
  - Action: Send credit/refund policy

✓ ENGINEERING BRIEFING:
  - Root cause: JWT audience mismatch
  - Fix location: JWT_AUDIENCE env var (prod)
  - Testing: Run auth-flow-e2e suite

✓ LEADERSHIP BRIEFING:
  - Financial exposure: ~$45K ARR
  - SLO impact: 99.95% → 98.2%
  - Decision: Declare P1, brief board

Each role got exactly what they need — no data digging required!
""")

## Step 4: Understanding Cognition by Context

### What Just Happened

You didn't write three separate briefing templates. Instead:

1. **You stored multi-faceted incident data** (technical, customer, support, engineering perspectives)
2. **You called Cognitive Gateway 3 times** with different role contexts
3. **Ninai's backend agents analyzed the SAME data** through different lenses:
   - GoalDecompositionAgent (Phase 21) — broke goal into role-specific actions
   - ContextAmplifierAgent (Phase 8) — amplified role-specific context
   - CognitiveGatewayService — routed to role-aware analysis
4. **You received 3 different, actionable briefings** tailored to each role

### The Architecture

```
┌─────────────────────────────┐
│ Single Incident Data        │
│ (Technical + Business +     │
│  Support + Engineering)     │
└──────────────┬──────────────┘
               │
        (Search by seed)
               │
     ┌─────────┴─────────┐
     │                   │
     ↓                   ↓
┌──────────────┐    ┌──────────────┐
│ Support Role │    │ Engineering  │  Leadership...
└──────┬───────┘    └──────┬───────┘
       │                   │
       ├→ Focus: Customer  │├→ Focus: Technical
       │  impact + ETA     ││  fix + validation
       │                   │
       └──────┬──────┬─────┘
              │      │
         (Ninai analyzes each through role lens)
              │      │
     ┌────────▼─┬────▼────────┐
     │           │             │
     ↓           ↓             ↓
Support Brief  Eng Brief    Leadership Brief
- 127 affected  - JWT bugfix  - $45K exposure
- ETA 30 min    - Env var fix - P1 declare
- Message draft - E2E test    - Board brief
```

### How Role Context Shapes Analysis

| Aspect | Support Focus | Engineering Focus | Leadership Focus |
|--------|---------------|-------------------|-----------------|
| **Priority** | Communication | Root cause | Business impact |
| **Metrics** | Affected customers | Debug info | Revenue/SLA |
| **Actions** | Message templates | Code changes | Escalation |
| **Confidence** | ETA certainty | Fix validity | Risk level |

### Key Insight: Same Data, Different Decisions

Demo 4 shows that **context matters**. Ninai's agents use the role parameter to:
- Reweight what's important
- Generate role-specific actions
- Surface relevant metrics
- Suggest tailored escalations

This is **contextual intelligence** — not just data retrieval.

### Next Steps

1. **Demo 5** — Multi-tenant data isolation (see how orgs stay separate)
2. **Cognitive Gateway Mastery** — Combine multiple verbs: write → search → decide → plan → explain
3. **Custom Roles** — Define your own role with custom focus areas

## Step 5: Troubleshooting — If Role-Based Briefings Didn't Work

In [ ]:
# Troubleshooting: Check SDK and backend
print("=" * 80)
print("TROUBLESHOOTING: Role-Based Briefing Generation")
print("=" * 80)

# Check 1: SDK has gateway.plan
print("\n[1] Checking SDK cognitive.gateway.plan()...")
try:
    if hasattr(client, 'cognitive') and hasattr(client.cognitive.gateway, 'plan'):
        print("OK: SDK has gateway.plan() method")
        import inspect
        sig = inspect.signature(client.cognitive.gateway.plan)
        print(f"    Signature: {sig}")
    else:
        print("ERROR: SDK missing gateway.plan()")
        print("  Fix: pip install -e ./repos/ninai/sdk/python")
except Exception as e:
    print(f"ERROR: {e}")

# Check 2: Verify incident memories exist
print("\n[2] Verifying incident data...")
try:
    verify = client.memories.search(query=f'seed={seed}', limit=10)
    if verify.items:
        print(f"OK: Found {len(verify.items)} incident memories")
        for mem in verify.items:
            facet = mem.content_preview.split(':')[0]
            print(f"    - {facet}")
    else:
        print("ERROR: No incident memories found")
        print("  Action: Go back to Step 2 and create memories")
except Exception as e:
    print(f"ERROR: {e}")

# Check 3: Test gateway.plan() call
print("\n[3] Testing gateway.plan() method...")
try:
    test_plan = client.cognitive.gateway.plan(
        goal="Test role-based planning",
        context={"role": "test", "test": True}
    )
    print("OK: gateway.plan() endpoint is responsive")
except AttributeError as e:
    print(f"ERROR: Method not available: {e}")
    print("  Fix: Ensure SDK 0.1.0+ with CognitiveGatewayResource")
except Exception as e:
    error_str = str(e)[:60]
    print(f"ERROR: Backend endpoint missing or unreachable: {error_str}")
    print("  Action: Start backend: cd repos/ninai/backend && uvicorn app.main:app")

print("\n" + "=" * 80)
print("DEMO 4 CHECKLIST")
print("=" * 80)
print("""
Prerequisites:
  [x] SDK 0.1.0+ installed (has plan() method)
  [x] Client authenticated with valid token
  [x] Incident data created in Step 2
  [?] Backend running at https://admin.ninai.sansten.com
  [?] /cognitive/gateway/plan endpoint accessible

If Step 3 failed:
  1. Check SDK has gateway.plan() method
  2. Verify incident memories were created (Step 2)
  3. Ensure backend is running
  4. Validate auth token is valid
  5. Check /cognitive/gateway/plan endpoint exists

Expected Output Format (per role):
{
    "goal": "...",
    "priority": "P1|P2|P3",
    "confidence": 0.85,
    "action_plan": [
        "Action 1 tailored to role",
        "Action 2 tailored to role",
        ...
    ],
    "focus_metrics": {
        "metric1": "value1",
        "metric2": "value2"
    },
    "reasoning": "Why this plan fits this role"
}

Each role gets a different briefing from the same incident data!
""")